# Phase 2 Tier 1: Review-Block Ablation on Colab

This notebook assumes you already ran the fast XGBoost importance notebook and produced:
- `phase2_tier1_xgb_importance.json`

It then:
- builds the Phase 2 feature manifest from the saved XGBoost importance output
- treats `accept + review` as the working set
- runs second-pass ablations on:
  - each review feature individually
  - each review block (`peak_flux`, `amplitude`, `timing`, `mean_flux`, `color`, `context`) when present
- compares each ablation against the working set on `PR-AUC` and `F1`
- writes the manifest and ablation report back to Google Drive

In [ ]:
%pip install -q xgboost pandas numpy

In [ ]:
import os
import json
import subprocess
import numpy as np
import pandas as pd
import xgboost as xgb
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
CSV_PATH = '/content/drive/MyDrive/supernovae_classification/data/processed/spcc_features_tier1.csv'
IMPORTANCE_JSON = '/content/drive/MyDrive/supernovae_classification/results/phase2_tier1/phase2_tier1_xgb_importance.json'
MANIFEST_PATH = '/content/drive/MyDrive/supernovae_classification/results/phase2_tier1/phase2_tier1_feature_manifest.json'
ABLATION_JSON = '/content/drive/MyDrive/supernovae_classification/results/phase2_tier1/phase2_tier1_review_ablation.json'
MODELS_DIR = '/content/drive/MyDrive/supernovae_classification/models/phase2_tier1/xgboost_ablation'

RANDOM_STATE = 42
TEST_SPLIT = 0.2
VALIDATION_SPLIT = 0.2
NUM_BOOST_ROUND = 250
EARLY_STOPPING_ROUNDS = 20

assert os.path.exists(CSV_PATH), f'Missing feature table: {CSV_PATH}'
assert os.path.exists(IMPORTANCE_JSON), f'Missing importance JSON: {IMPORTANCE_JSON}'
os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
os.makedirs(os.path.dirname(ABLATION_JSON), exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

print('CSV_PATH =', CSV_PATH)
print('IMPORTANCE_JSON =', IMPORTANCE_JSON)
print('MANIFEST_PATH =', MANIFEST_PATH)
print('ABLATION_JSON =', ABLATION_JSON)
try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
except Exception as exc:
    print('GPU check failed:', exc)

In [ ]:
with open(IMPORTANCE_JSON) as f:
    importance_payload = json.load(f)

full_features = importance_payload['full_baseline_features']
buckets = importance_payload['feature_buckets']
accept = buckets['accept']
review = buckets['review']
reject = buckets['reject']
working = accept + review

def block_members(feature_names, candidates):
    return [feature for feature in candidates if feature in feature_names]

review_blocks = {
    'peak_flux_block': block_members(review, [name for name in full_features if name.endswith('_peak_flux')] + ['peak_flux_all']),
    'amplitude_block': block_members(review, [name for name in full_features if name.endswith('_amplitude')] + ['amplitude_all']),
    'timing_block': block_members(review, [name for name in full_features if 'time_of_peak' in name] + ['time_span']),
    'mean_flux_block': block_members(review, [name for name in full_features if name.endswith('_mean_flux')] + ['mean_flux_all']),
    'color_block': block_members(review, [name for name in full_features if name.startswith('peak_color_')]),
    'context_block': block_members(review, ['observation_count', 'time_span', 'total_snr']),
}
review_blocks = {name: features for name, features in review_blocks.items() if features}

manifest = {
    'manifest_version': 'phase2_tier1_v1',
    'source_importance_file': IMPORTANCE_JSON,
    'importance_scope_note': importance_payload['importance_scope_note'],
    'full_baseline_features': full_features,
    'accept_features': accept,
    'review_features': review,
    'reject_features': reject,
    'working_feature_set': working,
    'top_permutation_features': [row['feature'] for row in importance_payload['permutation_importance'][:10]],
    'review_ablation_blocks': review_blocks,
    'drop_rule': {
        'primary_metric': 'pr_auc',
        'secondary_metric': 'f1',
        'guidance': 'Keep accept + review as the provisional working set. Drop only reject by default. Prune review features only through second-pass ablation.'
    },
}

with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print('Saved manifest:', MANIFEST_PATH)
print({k: len(v) for k, v in {'accept': accept, 'review': review, 'reject': reject}.items()})
display(pd.DataFrame({'bucket': ['accept', 'review', 'reject'], 'count': [len(accept), len(review), len(reject)]}))

In [ ]:
df = pd.read_csv(CSV_PATH)

def stratified_split_indices(labels, test_size, random_state):
    rng = np.random.default_rng(random_state)
    train_indices = []
    test_indices = []
    for label in np.unique(labels):
        label_indices = np.flatnonzero(labels == label)
        shuffled = label_indices.copy()
        rng.shuffle(shuffled)
        test_count = int(round(len(shuffled) * test_size))
        test_count = min(max(test_count, 1), len(shuffled) - 1)
        test_indices.extend(shuffled[:test_count])
        train_indices.extend(shuffled[test_count:])
    return np.array(sorted(train_indices)), np.array(sorted(test_indices))

def standardize(train_x, other_x):
    mean = train_x.mean(axis=0)
    std = train_x.std(axis=0)
    std[std == 0.0] = 1.0
    return (train_x - mean) / std, (other_x - mean) / std, mean, std

def roc_auc_score_numpy(y_true, scores):
    pos = scores[y_true == 1]
    neg = scores[y_true == 0]
    if len(pos) == 0 or len(neg) == 0:
        return 0.0
    comparisons = (pos[:, None] > neg[None, :]).sum()
    ties = (pos[:, None] == neg[None, :]).sum()
    return float((comparisons + 0.5 * ties) / (len(pos) * len(neg)))

def average_precision_numpy(y_true, scores):
    order = np.argsort(-scores)
    y_sorted = y_true[order]
    tp_cumsum = np.cumsum(y_sorted == 1)
    fp_cumsum = np.cumsum(y_sorted == 0)
    precision = tp_cumsum / np.maximum(tp_cumsum + fp_cumsum, 1)
    positive_total = max(int(np.sum(y_true == 1)), 1)
    recall = tp_cumsum / positive_total
    ap = 0.0
    previous_recall = 0.0
    for p_value, r_value, label in zip(precision, recall, y_sorted):
        if label == 1:
            ap += p_value * (r_value - previous_recall)
            previous_recall = r_value
    return float(ap)

def binary_metrics(y_true, probs, threshold=0.5):
    preds = (probs >= threshold).astype(np.int32)
    y_true = y_true.astype(np.int32)
    tp = int(np.sum((preds == 1) & (y_true == 1)))
    fp = int(np.sum((preds == 1) & (y_true == 0)))
    tn = int(np.sum((preds == 0) & (y_true == 0)))
    fn = int(np.sum((preds == 0) & (y_true == 1)))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    accuracy = (tp + tn) / len(y_true)
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc_score_numpy(y_true, probs),
        'pr_auc': average_precision_numpy(y_true, probs),
    }

def build_matrix(frame, feature_names):
    x = frame[feature_names].to_numpy(dtype=np.float32)
    y = (frame['label_name'] == 'Ia').astype(np.int32).to_numpy()
    return x, y

def base_xgb_params(scale_pos_weight):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'tree_method': 'hist',
        'verbosity': 0,
        'seed': RANDOM_STATE,
        'scale_pos_weight': scale_pos_weight,
    }
    try:
        params['device'] = 'cuda'
    except Exception:
        pass
    return params

def train_candidate_xgb(train_df, val_df, feature_names, params):
    x_train_raw, y_train = build_matrix(train_df, feature_names)
    x_val_raw, y_val = build_matrix(val_df, feature_names)
    x_train, x_val, _, _ = standardize(x_train_raw, x_val_raw)
    pos_count = float(np.sum(y_train == 1))
    neg_count = float(np.sum(y_train == 0))
    dtrain = xgb.DMatrix(x_train, label=y_train, feature_names=feature_names)
    dval = xgb.DMatrix(x_val, label=y_val, feature_names=feature_names)
    booster = xgb.train(
        params={**base_xgb_params(neg_count / max(pos_count, 1.0)), **params},
        dtrain=dtrain,
        num_boost_round=NUM_BOOST_ROUND,
        evals=[(dval, 'validation')],
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )
    probs = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
    return {'params': params, 'best_iteration': int(booster.best_iteration + 1), 'metrics': binary_metrics(y_val, probs)}

def fit_final_xgb(trainval_df, test_df, feature_names, params, num_boost_round):
    x_trainval_raw, y_trainval = build_matrix(trainval_df, feature_names)
    x_test_raw, y_test = build_matrix(test_df, feature_names)
    x_trainval, x_test, mean, std = standardize(x_trainval_raw, x_test_raw)
    pos_count = float(np.sum(y_trainval == 1))
    neg_count = float(np.sum(y_trainval == 0))
    dtrainval = xgb.DMatrix(x_trainval, label=y_trainval, feature_names=feature_names)
    dtest = xgb.DMatrix(x_test, label=y_test, feature_names=feature_names)
    booster = xgb.train(
        params={**base_xgb_params(neg_count / max(pos_count, 1.0)), **params},
        dtrain=dtrainval,
        num_boost_round=num_boost_round,
        verbose_eval=False,
    )
    probs = booster.predict(dtest)
    return booster, binary_metrics(y_test, probs)

labels = (df['label_name'] == 'Ia').astype(np.int32).to_numpy()
trainval_idx, test_idx = stratified_split_indices(labels, TEST_SPLIT, RANDOM_STATE)
train_idx_local, val_idx_local = stratified_split_indices(labels[trainval_idx], VALIDATION_SPLIT, RANDOM_STATE)
train_df = df.iloc[trainval_idx[train_idx_local]].reset_index(drop=True)
val_df = df.iloc[trainval_idx[val_idx_local]].reset_index(drop=True)
trainval_df = df.iloc[trainval_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

In [ ]:
working_features = manifest['working_feature_set']
candidate_runs = []
for params in [
    {'max_depth': 3, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1.0, 'lambda': 1.0},
    {'max_depth': 4, 'eta': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 1.0, 'lambda': 1.0},
    {'max_depth': 5, 'eta': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 2.0, 'lambda': 1.5},
]:
    candidate_runs.append(train_candidate_xgb(train_df, val_df, working_features, params))
best_run = max(candidate_runs, key=lambda item: item['metrics']['pr_auc'])

baseline_booster, baseline_metrics = fit_final_xgb(
    trainval_df, test_df, working_features, best_run['params'], best_run['best_iteration']
)

ablation_sets = {}
for feature in manifest['review_features']:
    ablation_sets[f'drop_feature__{feature}'] = [name for name in working_features if name != feature]
for block_name, block_features in manifest['review_ablation_blocks'].items():
    ablation_sets[f'drop_block__{block_name}'] = [name for name in working_features if name not in block_features]

def recommendation(pr_auc_delta, f1_delta):
    if pr_auc_delta >= -0.002 and f1_delta >= -0.002:
        return 'safe_to_drop'
    if pr_auc_delta <= -0.01 or f1_delta <= -0.01:
        return 'keep'
    return 'review_again'

ablation_rows = []
for name, feature_subset in ablation_sets.items():
    local_runs = []
    for params in [
        {'max_depth': 3, 'eta': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 1.0, 'lambda': 1.0},
        {'max_depth': 4, 'eta': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9, 'min_child_weight': 1.0, 'lambda': 1.0},
        {'max_depth': 5, 'eta': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 2.0, 'lambda': 1.5},
    ]:
        local_runs.append(train_candidate_xgb(train_df, val_df, feature_subset, params))
    local_best = max(local_runs, key=lambda item: item['metrics']['pr_auc'])
    _, metrics = fit_final_xgb(trainval_df, test_df, feature_subset, local_best['params'], local_best['best_iteration'])
    pr_auc_delta = metrics['pr_auc'] - baseline_metrics['pr_auc']
    f1_delta = metrics['f1'] - baseline_metrics['f1']
    ablation_rows.append({
        'name': name,
        'feature_count': len(feature_subset),
        'removed_features': [feature for feature in working_features if feature not in feature_subset],
        'metrics': metrics,
        'pr_auc_delta_vs_working_set': pr_auc_delta,
        'f1_delta_vs_working_set': f1_delta,
        'recommendation': recommendation(pr_auc_delta, f1_delta),
    })

ablation_rows.sort(key=lambda row: (row['pr_auc_delta_vs_working_set'], row['f1_delta_vs_working_set']), reverse=True)

output = {
    'source_manifest': MANIFEST_PATH,
    'baseline_working_set_metrics': baseline_metrics,
    'baseline_selection': best_run,
    'ablation_results': ablation_rows,
}

with open(ABLATION_JSON, 'w') as f:
    json.dump(output, f, indent=2)

print('Saved ablation report:', ABLATION_JSON)
print('Baseline working-set metrics:', baseline_metrics)
display(pd.DataFrame(ablation_rows).head(20))

## Interpretation rule

- `safe_to_drop`: negligible PR-AUC and F1 loss relative to the working set
- `keep`: clear degradation when removed
- `review_again`: ambiguous, likely correlated with another retained feature or block

Use this notebook to prune only within the `review` group. Keep `accept` unchanged unless later evidence contradicts it.